# Module 6: Wednesday Session
## CS 5045: Computation for the Data Sciences
### Object-Oriented Programming: Live Demo

**Goal for today:** Move from five loose operations scattered in a script to a clean,
organized class design where data and behavior live together.

We'll build the `Patient` and `Clinic` classes step by step, running each piece
before adding the next. We'll also see how **inheritance** lets a subclass extend
the Patient blueprint without rewriting it.

---

## Setup

Load the patient dataset so we have real data to work with.
This is the **Represent** operation at its simplest: raw data into memory.

In [ ]:
import pandas as pd

# REPRESENT: read the raw CSV into a pandas DataFrame.
# At this point every patient is just a row of values; there is no
# behavior attached to a row, only data. Turning this DataFrame into
# Patient objects (Step 1 below) is what adds behavior.
df = pd.read_csv('data.csv')

print(f"Loaded {len(df)} patients.")
df.head()   # peek at the first 5 rows to confirm the columns look right

---
## Step 1: Represent: Define the Patient Blueprint

Right now, each patient is just a row in a DataFrame, a set of values with no
behavior attached. A class lets us define what a patient **is** and what a patient
**can do**.

The `__init__` method runs automatically when we create a new Patient object.
`self` refers to the specific object being set up.

In [ ]:
class Patient:
    """
    Blueprint for one patient record.

    Every Patient object stores the same twelve attributes, listed in
    __init__ below. Defining __init__ is what makes Patient(...)
    callable: calling the class name like a function is how you ask
    Python to build a new object from this blueprint.
    """

    def __init__(self, patient_id, first_name, last_name,
                 age, sex, diagnosis,
                 systolic_bp, diastolic_bp, cholesterol,
                 bmi, smoker, num_visits):
        """
        Parameters:
            patient_id   (str)   -- unique identifier, e.g. 'P001'
            first_name   (str)   -- patient's first name
            last_name    (str)   -- patient's last name
            age          (int)   -- age in years; CSV values arrive as
                                     strings, so we convert with int()
            sex          (str)   -- 'F' or 'M'; stored as-is, no conversion needed
            diagnosis    (str)   -- 'Healthy', 'Hypertension', or 'Diabetes'
            systolic_bp  (int)   -- systolic blood pressure in mmHg
            diastolic_bp (int)   -- diastolic blood pressure in mmHg
            cholesterol  (int)   -- total cholesterol, mg/dL
            bmi          (float) -- body mass index; needs float(), not
                                     int(), because BMI is naturally a
                                     decimal value
            smoker       (bool)  -- True/False; the CSV stores 'yes'/'no'
                                     as text, so we convert with a string
                                     comparison, not a type-cast
            num_visits   (int)   -- number of clinic visits on record

        Every value above arrives from the CSV as a string. Converting
        types here, once, means every other method on this class can
        trust that self.age is a real int and self.bmi is a real float,
        rather than re-checking or re-converting types everywhere they
        get used later (see is_high_risk in Step 2 for exactly that).
        """
        # REPRESENT: store each attribute on the object, converting
        # types as we go so downstream math (e.g. is_high_risk below)
        # works without extra conversions.
        self.patient_id   = patient_id
        self.first_name   = first_name
        self.last_name    = last_name
        self.age          = int(age)
        self.sex          = sex
        self.diagnosis    = diagnosis
        self.systolic_bp  = int(systolic_bp)
        self.diastolic_bp = int(diastolic_bp)
        self.cholesterol  = int(cholesterol)
        self.bmi          = float(bmi)
        # str(...).lower() == 'yes' turns any of 'yes' / 'Yes' / 'YES'
        # into a real Python bool, rather than storing the original
        # string as-is.
        self.smoker       = str(smoker).lower() == 'yes'
        self.num_visits   = int(num_visits)


print("Patient class defined: no objects created yet.")
# Defining a class never creates any data on its own. Nothing "happens"
# until we call Patient(...) below, the same way defining a function
# never runs its body until you actually call it.

In [ ]:
# Create one Patient object from Alice's row.
# Calling Patient(...) triggers __init__ above: Python allocates a new,
# empty object, then runs __init__ with that new object bound to `self`.
alice = Patient(
    patient_id='P001', first_name='Alice', last_name='Morrow',
    age=54, sex='F', diagnosis='Hypertension',
    systolic_bp=148, diastolic_bp=92, cholesterol=210,
    bmi=27.4, smoker='no', num_visits=6
)

# Access attributes with dot notation: object.attribute_name.
# No parentheses here; these are stored values, not method calls.
print(f"ID:        {alice.patient_id}")
print(f"Name:      {alice.first_name} {alice.last_name}")
print(f"Age:       {alice.age}")
print(f"Diagnosis: {alice.diagnosis}")
print(f"Sys BP:    {alice.systolic_bp} mmHg")
print(f"Smoker:    {alice.smoker}")  # stored as True/False, not 'no'

---
## Step 2: Transform: Add Methods to the Patient

A method is a function that lives inside a class and has access to `self`,
meaning it can use the object's own data without needing it passed as an argument.

**Pause and ask:** Before I call `is_high_risk()`: who thinks Alice is high risk?
What is your reasoning?

In [ ]:
class Patient:
    """Blueprint for one patient record, now with Transform methods."""

    def __init__(self, patient_id, first_name, last_name,
                 age, sex, diagnosis,
                 systolic_bp, diastolic_bp, cholesterol,
                 bmi, smoker, num_visits):
        """See Step 1 above for the full parameter list and rationale
        for each type conversion; unchanged here."""
        self.patient_id   = patient_id
        self.first_name   = first_name
        self.last_name    = last_name
        self.age          = int(age)
        self.sex          = sex
        self.diagnosis    = diagnosis
        self.systolic_bp  = int(systolic_bp)
        self.diastolic_bp = int(diastolic_bp)
        self.cholesterol  = int(cholesterol)
        self.bmi          = float(bmi)
        self.smoker       = str(smoker).lower() == 'yes'
        self.num_visits   = int(num_visits)

    # TRANSFORM: every method below computes a new value from this
    # patient's own attributes. None of them change self.* in place;
    # they read existing data and return something new each time they
    # are called.

    def full_name(self):
        """
        TRANSFORM: combine first and last name into one display string.

        Parameters: none (uses self.first_name and self.last_name)
        Returns: str -- "First Last"
        """
        return f"{self.first_name} {self.last_name}"

    def pulse_pressure(self):
        """
        TRANSFORM: systolic minus diastolic blood pressure.

        Parameters: none (uses self.systolic_bp and self.diastolic_bp)
        Returns: int -- pulse pressure in mmHg. Values above 60 mmHg
                 are clinically considered elevated.
        """
        return self.systolic_bp - self.diastolic_bp

    def is_high_risk(self):
        """
        TRANSFORM: derive a single Boolean risk flag from three criteria.

        Parameters: none (uses self.systolic_bp, self.bmi, self.smoker,
                    and self.diagnosis)
        Returns: bool -- True if ANY of the three criteria below hold:
                 1. hypertensive: systolic_bp >= 140
                 2. obese:        bmi >= 30
                 3. risky_smoker: a smoker whose diagnosis is not
                    'Healthy' (a smoker with no diagnosed condition yet
                    is not flagged by this rule alone)
                 The three booleans are combined with `or`, so any one
                 being True makes the whole patient high risk.
        """
        hypertensive = self.systolic_bp >= 140
        obese        = self.bmi >= 30
        risky_smoker = self.smoker and self.diagnosis != 'Healthy'
        return hypertensive or obese or risky_smoker

    def risk_label(self):
        """
        TRANSFORM: turn the Boolean from is_high_risk() into a human-
        readable string.

        Parameters: none (calls self.is_high_risk())
        Returns: str -- 'High Risk' or 'Standard Risk'
        """
        return 'High Risk' if self.is_high_risk() else 'Standard Risk'

    # Communicate: control how the object displays itself
    def __repr__(self):
        """
        COMMUNICATE: control what this object looks like when printed
        or displayed. Python calls __repr__ automatically whenever an
        object appears in a print() call, an f-string, or a raw
        notebook cell's output; we never call it directly ourselves.

        Parameters: none
        Returns: str -- a one-line summary combining several other
                 methods (full_name, risk_label) with raw attributes.
        """
        return (f"Patient({self.patient_id}, {self.full_name()}, "
                f"age={self.age}, {self.diagnosis}, {self.risk_label()})")


print("Patient class redefined with Transform methods.")

In [ ]:
# Re-create Alice with the updated class, and add a second patient
# (Ben) so we have more than one object to compare.
alice = Patient('P001','Alice','Morrow',54,'F','Hypertension',148,92,210,27.4,'no',6)
ben   = Patient('P002','Ben','Okafor',41,'M','Diabetes',132,85,245,31.2,'yes',12)

# TRANSFORM: call methods on each object. Note the parentheses; these
# are method calls (run code, return a value), not attribute lookups.
print("Alice:")
print(f"  Pulse pressure: {alice.pulse_pressure()} mmHg")
print(f"  High risk?      {alice.is_high_risk()}")
print(f"  Risk label:     {alice.risk_label()}")
print()
print("Ben:")
print(f"  Pulse pressure: {ben.pulse_pressure()} mmHg")
print(f"  High risk?      {ben.is_high_risk()}")
print(f"  Risk label:     {ben.risk_label()}")
print()
# COMMUNICATE: printing an object directly triggers __repr__ behind
# the scenes. We never wrote print(alice.__repr__()); Python does
# that lookup for us automatically.
print(alice)
print(ben)

---
## Step 3: Inheritance: Extending the Patient Blueprint

Some patients need extra tracking: a dedicated care manager, a priority routing flag.
We could copy the Patient class and add new fields, but then we have two classes
to maintain forever.

**Inheritance** solves this. `HighRiskPatient(Patient)` starts with everything
Patient already has. We only write what is new or different.

Key ideas:
- `class HighRiskPatient(Patient):`: the parentheses name the parent class
- `super().__init__(...)`: calls the parent's `__init__` so all Patient attributes are set up
- One new attribute (`care_manager`): added after `super().__init__()` returns
- One overridden method (`risk_label`): replaces the parent version for this subclass only
- One new method (`priority_alert`): only exists on HighRiskPatient, not on Patient
- Everything else (pulse_pressure, is_high_risk, full_name, __repr__) is **inherited**: we
  never wrote it here, but it works.

In [ ]:
class HighRiskPatient(Patient):
    """
    A Patient with elevated risk requiring dedicated care management.

    Inherits all Patient attributes and methods (full_name,
    pulse_pressure, is_high_risk, __repr__, everything) without
    rewriting any of them.

    Adds:      care_manager attribute
    Overrides: risk_label() -- always returns the priority label
    New method: priority_alert()
    """

    def __init__(self, patient_id, first_name, last_name,
                 age, sex, diagnosis,
                 systolic_bp, diastolic_bp, cholesterol,
                 bmi, smoker, num_visits,
                 care_manager):                        # <-- one new parameter
        """
        Parameters:
            (all Patient parameters, unchanged -- see Step 1)
            care_manager (str) -- name of the clinician assigned to
                                   coordinate this patient's care; the
                                   one attribute Patient does not have

        super().__init__(...) hands every Patient-shaped parameter up
        to the parent class's __init__, so self.age, self.bmi,
        self.smoker, etc. all get set up exactly as they would for a
        plain Patient. We only need to write code here for what is
        new: care_manager.
        """
        # Call the parent __init__ to set up all Patient attributes.
        # Without this line, self.age, self.bmi, etc. would not exist,
        # and every inherited method that reads them would raise
        # AttributeError the moment it ran.
        super().__init__(patient_id, first_name, last_name,
                         age, sex, diagnosis,
                         systolic_bp, diastolic_bp, cholesterol,
                         bmi, smoker, num_visits)
        # Now add the one attribute specific to high-risk patients.
        # This line only runs AFTER super().__init__() above has
        # finished, so all the Patient attributes already exist by
        # the time we get here.
        self.care_manager = care_manager

    def risk_label(self):
        """
        OVERRIDE of Patient.risk_label(): HighRiskPatient is always
        labeled as priority, regardless of what is_high_risk() would
        say. Overriding means Python looks up risk_label() on
        HighRiskPatient first and finds this version; the parent's
        version is never reached for HighRiskPatient objects.

        Parameters: none
        Returns: str -- the literal string 'High Risk (Priority)'
        """
        return 'High Risk (Priority)'

    def priority_alert(self):
        """
        NEW method: exists only on HighRiskPatient, not on Patient.
        Produces a care management notice combining an inherited
        method (full_name) with this subclass's own attribute
        (care_manager).

        Parameters: none
        Returns: str -- a formatted alert naming the patient, their
                 assigned care manager, and their clinical details.
        """
        return (
            f"{self.full_name()}, age {self.age}: "
            f"assigned to {self.care_manager}. "
            f"Diagnosis: {self.diagnosis}, systolic BP: {self.systolic_bp}."
        )


print("HighRiskPatient class defined.")

In [ ]:
# Create a HighRiskPatient using Alice's clinical data, plus the one
# new field (care_manager) that only this subclass requires.
alice_hr = HighRiskPatient(
    patient_id='P001', first_name='Alice', last_name='Morrow',
    age=54, sex='F', diagnosis='Hypertension',
    systolic_bp=148, diastolic_bp=92, cholesterol=210,
    bmi=27.4, smoker='no', num_visits=6,
    care_manager='Dr. Chen'                            # the one new field
)

# OVERRIDDEN method: HighRiskPatient's own risk_label() runs here, not
# Patient's version, even though both classes define a method with
# this exact name.
print("risk_label()       :", alice_hr.risk_label())

# NEW method: only defined on HighRiskPatient. Calling this on a plain
# Patient object would raise AttributeError (demonstrated below).
print("priority_alert()   :", alice_hr.priority_alert())

print()
# INHERITED methods -- we never wrote these in HighRiskPatient
# We never wrote pulse_pressure, is_high_risk, or full_name anywhere
# in HighRiskPatient. Python looks for the method on HighRiskPatient
# first, doesn't find it, then checks the parent class (Patient) and
# finds it there. This automatic fallback is what "inheriting" a
# method means.
print("pulse_pressure()   :", alice_hr.pulse_pressure(), "mmHg  (inherited from Patient)")
print("is_high_risk()     :", alice_hr.is_high_risk(), "  (inherited from Patient)")
print("full_name()        :", alice_hr.full_name(), "       (inherited from Patient)")
print()

# __repr__ is inherited too: notice risk_label() is now 'High Risk (Priority)'
# __repr__ is inherited unchanged from Patient, but the TEXT it
# produces differs, because its f-string calls self.risk_label(), and
# for a HighRiskPatient object that resolves to the overridden
# version, not Patient's.
print("repr:", alice_hr)

print()
# Confirm that a plain Patient does NOT have priority_alert
# This is the payoff of inheritance being one-directional:
# HighRiskPatient gets everything Patient has, but Patient never gets
# anything back from its subclasses.
alice_plain = Patient('P001','Alice','Morrow',54,'F','Hypertension',148,92,210,27.4,'no',6)
print("Plain Patient risk_label():", alice_plain.risk_label())   # 'High Risk' (parent version)
try:
    alice_plain.priority_alert()
except AttributeError as e:
    print(f"Plain Patient has no priority_alert: {e}")

---
## Step 4: Represent: Load All Patients from CSV

Now we turn every row of the DataFrame into a Patient object.
The result is a Python list of sixteen Patient objects: structured,
typed, and ready for further operations.

**Pause and ask:** After we load, how would you check that the
first patient's age stored correctly?

In [ ]:
def load_patients(filepath):
    """
    REPRESENT: read a CSV file and build one Patient object per row.

    Parameters:
        filepath (str) -- path to a CSV file with the same columns as
                           data.csv (patient_id, first_name, last_name,
                           age, sex, diagnosis, systolic_bp,
                           diastolic_bp, cholesterol, bmi, smoker,
                           num_visits)

    Returns:
        list of Patient -- one object per row, in the same order as
        the file. Patient.__init__ handles all the string-to-int /
        float / bool conversions, so this function doesn't need to
        convert anything itself; it just hands each row's values to
        Patient(...) and lets the class do that work.
    """
    df = pd.read_csv(filepath)
    patients = []
    # iterrows() yields (index, row) pairs; we only need the row, so
    # the index is thrown away with the underscore variable name.
    for _, row in df.iterrows():
        p = Patient(
            patient_id   = row['patient_id'],
            first_name   = row['first_name'],
            last_name    = row['last_name'],
            age          = row['age'],
            sex          = row['sex'],
            diagnosis    = row['diagnosis'],
            systolic_bp  = row['systolic_bp'],
            diastolic_bp = row['diastolic_bp'],
            cholesterol  = row['cholesterol'],
            bmi          = row['bmi'],
            smoker       = row['smoker'],
            num_visits   = row['num_visits'],
        )
        patients.append(p)
    return patients

patients = load_patients('data.csv')
print(f"Loaded {len(patients)} Patient objects.")
print()
# COMMUNICATE: print shows __repr__ for each object
# print(p) triggers Patient.__repr__ for each object, the same
# mechanism from Step 2, now running inside a loop.
for p in patients[:4]:
    print(p)

---
## Step 5: Filter: The Clinic Collection Class

Individual patients are useful. But most data science questions are about groups.
A `Clinic` class manages the collection and provides Filter and Aggregate operations
that work across all patients.

**Pause and ask:** Before I run `high_risk()`: how many patients
do you expect to come back? What is your reasoning?

In [ ]:
class Clinic:
    """
    Manages a collection of Patient objects.

    Where Patient represents ONE record, Clinic represents MANY: it
    holds a list of Patient objects and provides operations that work
    across the whole group (Filter, and later Aggregate), the kind of
    question a single Patient object cannot answer on its own.
    """

    def __init__(self, name='General Clinic'):
        """
        Parameters:
            name (str) -- display name for this clinic; defaults to
                          'General Clinic' if not given

        A new Clinic always starts with an empty patients list.
        Patients are added one at a time with add_patient() below;
        they are never passed in here.
        """
        self.name     = name
        self.patients = []

    def add_patient(self, patient):
        """
        REPRESENT: add one Patient object to this clinic's collection.

        Parameters:
            patient (Patient) -- the Patient object to add
        Returns: None (modifies self.patients in place)
        """
        self.patients.append(patient)

    def size(self):
        """
        AGGREGATE: count how many patients are in this clinic.

        Parameters: none
        Returns: int -- the number of Patient objects currently stored
        """
        return len(self.patients)

    # Filter methods: return subsets
    # Each one scans self.patients and returns a smaller list
    # containing only the patients that meet one condition. None of
    # these methods change self.patients; they read it and build a
    # new list.

    def by_diagnosis(self, diagnosis):
        """
        FILTER: keep only patients with a specific diagnosis.

        Parameters:
            diagnosis (str) -- exact diagnosis string to match, e.g.
                                'Diabetes' (case-sensitive)
        Returns: list of Patient -- the matching subset, in original order
        """
        return [p for p in self.patients if p.diagnosis == diagnosis]

    def high_risk(self):
        """
        FILTER: keep only patients whose is_high_risk() is True.

        Parameters: none
        Returns: list of Patient -- the high-risk subset. This method
                 doesn't recompute risk itself; it delegates to each
                 Patient's own is_high_risk() method and just collects
                 the ones that return True.
        """
        return [p for p in self.patients if p.is_high_risk()]

    def older_than(self, age_threshold):
        """
        FILTER: keep only patients strictly older than a given age.

        Parameters:
            age_threshold (int) -- patients with age > this value are kept
        Returns: list of Patient -- the matching subset
        """
        return [p for p in self.patients if p.age > age_threshold]

print("Clinic class defined.")

In [ ]:
# Build the clinic from our loaded patient list: one Clinic object,
# populated by calling add_patient() sixteen times (once per patient
# already created by load_patients() above).
clinic = Clinic(name='VT Teaching Clinic')
for p in patients:
    clinic.add_patient(p)

print(f"Clinic: {clinic.name}")
print(f"Total patients: {clinic.size()}")

# Filter: high-risk patients
# high_risk() scans all sixteen patients and returns only the ones
# whose own is_high_risk() method returns True.
high_risk = clinic.high_risk()
print(f"\nHigh-risk patients: {len(high_risk)}")
for p in high_risk:
    print(f"  {p}")

---
## Step 6: Aggregate: Summarizing Across Groups

Filter returns a list. Aggregate collapses a list into one number.
These two operations compose naturally: filter first, then aggregate the result.

**Pause and ask:** After we filter to diabetics, what mean systolic BP
do you expect, relative to the healthy group?

In [ ]:
# Add Aggregate methods to Clinic (redefine the class completely: the
# same pattern used for Patient in Step 2, a fresh class definition
# that includes everything from before, plus new methods).
class Clinic:
    """
    Manages a collection of Patient objects, now with Aggregate methods.

    Represent + Filter methods are unchanged from Step 5; this version
    adds methods that collapse a list of patients down to one number
    or one summary dict (Aggregate), plus one method that prints
    everything together (Communicate).
    """

    def __init__(self, name='General Clinic'):
        """See Step 5 for parameter details; unchanged here."""
        self.name     = name
        self.patients = []

    def add_patient(self, patient):
        """REPRESENT: add one Patient object. See Step 5 for details."""
        self.patients.append(patient)

    def size(self):
        """AGGREGATE: count of patients. See Step 5 for details."""
        return len(self.patients)

    def by_diagnosis(self, diagnosis):
        """FILTER: patients matching one diagnosis. See Step 5."""
        return [p for p in self.patients if p.diagnosis == diagnosis]

    def high_risk(self):
        """FILTER: patients flagged high risk. See Step 5."""
        return [p for p in self.patients if p.is_high_risk()]

    def older_than(self, age_threshold):
        """FILTER: patients older than a threshold. See Step 5."""
        return [p for p in self.patients if p.age > age_threshold]

    # Aggregate methods: collapse a group to one number
    # Each one collapses a group of patients down to a single number
    # or a single summary structure (dict).

    def mean_systolic_bp(self, subset=None):
        """
        AGGREGATE: average systolic BP, optionally over a subset.

        Parameters:
            subset (list of Patient, optional) -- if given, average
                    only over this group (e.g. the output of a Filter
                    method like by_diagnosis()); if omitted, average
                    over every patient in the clinic
        Returns: float -- the mean systolic BP, or None if the group
                 is empty (guards against a divide-by-zero on
                 len(group))
        """
        group = subset if subset is not None else self.patients
        if not group:
            return None
        return sum(p.systolic_bp for p in group) / len(group)

    def mean_bmi(self, subset=None):
        """
        AGGREGATE: average BMI, optionally over a subset.

        Parameters:
            subset (list of Patient, optional) -- same meaning as in
                    mean_systolic_bp() above
        Returns: float -- the mean BMI, or None if the group is empty
        """
        group = subset if subset is not None else self.patients
        if not group:
            return None
        return sum(p.bmi for p in group) / len(group)

    def count_by_diagnosis(self):
        """
        AGGREGATE: count patients per diagnosis category.

        Parameters: none
        Returns: dict -- {diagnosis_string: count}, one entry per
                 distinct diagnosis present in self.patients. Built
                 with .get(key, 0) so a diagnosis seen for the first
                 time starts at 0 before being incremented, rather
                 than raising a KeyError.
        """
        counts = {}
        for p in self.patients:
            counts[p.diagnosis] = counts.get(p.diagnosis, 0) + 1
        return counts

    def risk_summary(self):
        """
        AGGREGATE: count high-risk vs. standard-risk patients.

        Parameters: none
        Returns: dict -- {'High Risk': int, 'Standard Risk': int},
                 always both keys present even if one count is zero
                 (unlike count_by_diagnosis, which only includes
                 diagnoses actually seen)
        """
        summary = {'High Risk': 0, 'Standard Risk': 0}
        for p in self.patients:
            label = 'High Risk' if p.is_high_risk() else 'Standard Risk'
            summary[label] += 1
        return summary

    # Communicate: print a full summary report
    def print_summary(self):
        """
        COMMUNICATE: print a full summary report for this clinic.

        Parameters: none
        Returns: None (prints directly to stdout)

        Calls several of the Aggregate methods above (size,
        mean_systolic_bp, mean_bmi, count_by_diagnosis, risk_summary)
        and formats their results into one readable block, rather than
        making the caller call each one separately and assemble the
        output themselves.
        """
        print(f"\n=== {self.name}: Summary Report ===")
        print(f"Total patients   : {self.size()}")
        print(f"Mean systolic BP : {self.mean_systolic_bp():.1f} mmHg")
        print(f"Mean BMI         : {self.mean_bmi():.1f}")
        print("\nBy diagnosis:")
        for diag, count in sorted(self.count_by_diagnosis().items()):
            print(f"  {diag:<15} {count}")
        print("\nRisk distribution:")
        for label, count in self.risk_summary().items():
            print(f"  {label:<15} {count}")

print("Clinic redefined with Aggregate methods.")

In [ ]:
# Rebuild the clinic with the updated class: the OLD clinic object
# above was built from the Step 5 version of Clinic, which has no
# Aggregate methods. Re-running this loop against the new class
# definition gives us a clinic object that actually has mean_bmi(),
# print_summary(), etc.
clinic = Clinic(name='VT Teaching Clinic')
for p in patients:
    clinic.add_patient(p)

# Filter + Aggregate: mean BP by diagnosis group
# For each diagnosis, first Filter down to just that group
# (by_diagnosis), then Aggregate that group alone (mean_systolic_bp
# with subset=group). This is the same two-step pattern from the
# Attempt segment's Task 4.
print("Mean Systolic BP by Diagnosis:")
for diag in ['Healthy', 'Hypertension', 'Diabetes']:
    group   = clinic.by_diagnosis(diag)             # Filter
    mean_bp = clinic.mean_systolic_bp(subset=group)  # Aggregate
    print(f"  {diag:<15} n={len(group)}  mean BP = {mean_bp:.1f} mmHg")

print()
# Aggregate: Transform each patient's pulse pressure, then display
# This loop is actually Transform, not Aggregate: pulse_pressure() is
# computed once PER PATIENT, it never collapses the list of patients
# into one number. Contrast this with mean_systolic_bp above, which
# does collapse many values into one. This just displays the
# per-patient Transform result for all sixteen patients side by side.
print("Pulse Pressure per Patient:")
print(f"{'Name':<22} {'PP (mmHg)':>10}  Flag")
print("-" * 45)
for p in patients:
    pp   = p.pulse_pressure()               # Transform on individual
    flag = ' *** elevated' if pp > 60 else ''
    print(f"{p.full_name():<22} {pp:>10}{flag}")

In [ ]:
# Communicate: full summary report
# One call runs every Aggregate method defined above and prints a
# single readable report, instead of the caller assembling the pieces
# by hand.
clinic.print_summary()

---
## Connecting to the Assignment

The assignment (hw6.ipynb) uses the same patient dataset and the same two-level pattern.
Here is the design checklist:

**Patient class:**
- `__init__`: store five key attributes (age, diagnosis, systolic_bp, bmi, smoker) with correct type conversions
- `is_high_risk()`: return True if systolic_bp > 140, OR bmi > 30, OR smoker is True
- `__repr__`: a human-readable string describing the patient

**Clinic class:**
- `__init__`: initialize a patients list
- `average_bmi()`: compute the mean BMI across all patients in the collection

**Inheritance is not required in hw6**, but it will appear on the midterm.
Keep this notebook as your reference for the pattern.

**Design questions to answer before writing code:**
1. What type does each attribute need? (int, float, bool)
2. What does `smoker == 'yes'` evaluate to? (True or False)
3. How do you combine three conditions with `or`?
4. How do you compute a mean from a list of objects?

When those four questions are answered, the code follows directly.